In [ ]:
#!pip install gurobipy

---
We are operating on two variables:

xij - binary variable that decides if we are taking the path

fi - amount of fuel after getting to the destination

---
Goal function would be

min SIGMA(ij) dij * xij,

where dij is the distance between two points

---


# GETTING READY

In [ ]:
# data

import random

# n = 15
MAP_SIZE = 200

# cities = []

# for _ in range(n):
#   x = random.randint(0, MAP_SIZE)
#   y = random.randint(0, MAP_SIZE)
#   cities.append((x, y))

# print(f"Cities: {cities}")

cities = [
  (41, 120),
  (90, 101),
  (12, 13),
  (200, 140),
  (196, 182),
  (155, 71),
  (0, 18),
  (185, 125),
  (183, 0),
  (22, 71),
  (104, 142),
  (158, 93),
  (47, 180),
  (151, 166),
  (76, 198),
]
n = len(cities)

FUEL_CAPACITY = 2000

In [ ]:
# distance matrix

import math

dist = {}

for i in range(n):
  for j in range(n):
    if i != j:
      x1, y1 = cities[i]
      x2, y2 = cities[j]

      d = math.sqrt((x1-x2)**2 + (y1-y2)**2)
      dist[i, j] = round(d, 2)

# MODEL

In [ ]:
from gurobipy import *

m = Model("TSPWR")

# xij
x = m.addVars(dist.keys(), vtype=GRB.BINARY, name="x")

# Miller-Tucker-Zemlin formulation
u = m.addVars(n, lb=0, ub=n-1, vtype=GRB.CONTINUOUS, name="u")

# fuel after arrival
fuel = m.addVars(n, lb=0, ub=FUEL_CAPACITY, vtype=GRB.CONTINUOUS, name="fuel")

In [ ]:
# goal function

m.setObjective(quicksum(dist[i, j] * x[i, j] for i, j in dist), GRB.MINIMIZE)

# CONSTRAINTS

In [ ]:
# each city left once
for i in range(n):
  m.addConstr(quicksum(x[i, j] for j in range(n) if i != j) == 1)

# each city entered once
for j in range(n):
  m.addConstr(quicksum(x[i, j] for i in range(n) if i != j) == 1)

In [ ]:
# MTZ (no subtours)

for i in range(1, n):
  for j in range(1, n):
    if i != j:
      m.addConstr(u[i] - u[j] + n * x[i, j] <= n - 1)

In [ ]:
# fuel

# we have to calculate fuel in point 0 separately
# and avoid it in the for loop
# as when the fuel for coming back to the point 0 was calculated
# the constraint setting it to FUEL_CAPACITY was contradicting other constraints

m.addConstr(fuel[0] == FUEL_CAPACITY)

M_VAL = FUEL_CAPACITY + round(max(dist.values()))

for i, j in dist:
  if j != 0:
    m.addConstr(fuel[j] <= fuel[i] - dist[i, j] + M_VAL * (1 - x[i, j]))
    m.addConstr(fuel[j] >= fuel[i] - dist[i, j] - M_VAL * (1 - x[i, j]))

In [ ]:
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 605 rows, 240 columns and 2143 nonzeros (Min)
Model fingerprint: 0x25089a67
Model has 210 linear objective coefficients
Variable types: 30 continuous, 210 integer (210 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 3e+02]
  Bounds range     [1e+00, 2e+03]
  RHS range        [1e+00, 3e+03]

Presolve removed 1 rows and 2 columns
Presolve time: 0.02s
Presolved: 604 rows, 238 columns, 4238 nonzeros
Variable types: 28 continuous, 210 integer (210 binary)
Found heuristic solution: objective 1884.0300000
Found heuristic solution: objective 1799.0300000
Found heuristic solution: objective 1795.9800000

Root relaxation: objective 6.359074e+02, 87 iterations, 0.00 seconds (0.00 work units)

    Nod

# SOLUTION

In [ ]:
if m.status == GRB.OPTIMAL:
  print("Route:")

  for i, j in dist:
    if x[i, j].X > 0.5:
      print(f"{i} -> {j}")

  print("\nCost: ", round(m.objVal, 2))

  print("\nFuel: ")
  for i in range(n):
    print(i, round(fuel[i].X, 2))
else:
  print("No solution!")

Route:
0 -> 1
1 -> 10
2 -> 6
3 -> 7
4 -> 3
5 -> 8
6 -> 9
7 -> 11
8 -> 2
9 -> 0
10 -> 12
11 -> 5
12 -> 14
13 -> 4
14 -> 13

Cost:  826.02

Fuel: 
0 2000.0
1 1947.45
2 1296.91
3 1630.0
4 1672.19
5 1544.72
6 1283.91
7 1608.79
8 1468.4
9 1226.53
10 1904.13
11 1566.92
12 1835.62
13 1719.95
14 1801.49


In [ ]:
gurobi_time = m.Runtime
print("Gurobi time (s):", gurobi_time)

Gurobi time (s): 0.6344180107116699
